# Proyecto Integral: Análisis del Tráfico Aéreo SFO ✈️

Este notebook implementa todas las fases del proyecto:
1. Preparación y EDA
2. Clustering (Segmentación de Aerolíneas)
3. Modelos Predictivos: Regresión (pasajeros) y Clasificación (tipo operativo)
4. Series Temporales (Pronóstico mensual SARIMA)
5. Visualizaciones avanzadas y Recomendaciones

Dataset esperado: `air_traffic_data.csv` (estructura similar al de Kaggle: Air Traffic Passenger Statistics).

## 0. Configuración e Importaciones
Incluye instalación opcional de librerías adicionales si no están presentes en el entorno.

In [ ]:
# !pip install pandas numpy seaborn matplotlib scikit-learn statsmodels plotly pmdarima tqdm shap prophet folium

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, confusion_matrix

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    import pmdarima as pm
    PMDARIMA_AVAILABLE = True
except Exception:
    PMDARIMA_AVAILABLE = False

plt.style.use('seaborn-v0_8')

DATA_PATH = 'air_traffic_data.csv'  # Ajustar si es necesario

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'No se encontró el archivo {DATA_PATH}. Por favor colocarlo en el directorio de trabajo.')

## 1. Carga y Exploración Inicial
Se estandarizan nombres y se inspecciona la estructura.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print('Dimensiones iniciales:', raw.shape)
raw.head()

In [ ]:
# Copia de trabajo

df = raw.copy()

# Normalización de nombres de columnas
rename_map = {c: c.strip().lower().replace(' ', '_') for c in df.columns}
df.rename(columns=rename_map, inplace=True)

# Intento de identificar columnas clave (asunciones basadas en dataset típico)
# Columnas esperadas: operating_airline, activity_type_code, passenger_count, geo_region, terminal, boarding_area, date
# A veces existen columnas de Year/Month en lugar de date.

if 'date' in df.columns:
    try:
        df['date'] = pd.to_datetime(df['date'])
    except Exception:
        pass
else:
    # Crear fecha si hay year y month
    year_col = [c for c in df.columns if 'year' in c]
    month_col = [c for c in df.columns if 'month' in c]
    if year_col and month_col:
        df['date'] = pd.to_datetime(df[year_col[0]].astype(str) + '-' + df[month_col[0]].astype(str) + '-01')
    else:
        print('No se pudo construir columna fecha (date). Algunas funciones de series temporales no estarán disponibles.')

# Convertir passenger_count a numérico
if 'passenger_count' in df.columns:
    df['passenger_count'] = pd.to_numeric(df['passenger_count'], errors='coerce')

# Eliminar filas completamente vacías
before = df.shape[0]
df.dropna(how='all', inplace=True)

# Tratamiento de nulos en variables críticas
critical_cols = ['operating_airline','activity_type_code','passenger_count']
for col in critical_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            if col == 'passenger_count':
                df[col].fillna(0, inplace=True)
            else:
                df[col].fillna('Unknown', inplace=True)

# Eliminar duplicados
dup_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)

print(f"Filas eliminadas por estar vacías: {before - df.shape[0]}")
print(f"Duplicados eliminados: {dup_before}")
print('\nInfo:')
print(df.info())

In [ ]:
# Descriptivo básico
desc = df.describe(include='all').transpose()
desc.head(15)

### 1.1 Métricas Clave
Top aerolíneas, regiones y terminales.

In [ ]:
# Top 10 aerolíneas
if 'operating_airline' in df.columns and 'passenger_count' in df.columns:
    top_airlines = (df.groupby('operating_airline')['passenger_count']
                      .sum().sort_values(ascending=False).head(10))
    display(top_airlines.to_frame('total_pasajeros'))

# Top 10 regiones
if 'geo_region' in df.columns:
    top_regions = (df.groupby('geo_region')['passenger_count']
                     .sum().sort_values(ascending=False).head(10))
    display(top_regions.to_frame('total_pasajeros'))

# Visualización
fig, ax = plt.subplots(figsize=(10,5))
(top_airlines/1e6).plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Pasajeros (Millones)')
ax.set_title('Top 10 Aerolíneas por Pasajeros')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Pasajeros por terminal
if 'terminal' in df.columns:
    term_pass = df.groupby('terminal')['passenger_count'].sum().sort_values(ascending=False)
    sns.barplot(x=term_pass.index, y=term_pass.values, palette='viridis')
    plt.title('Distribución de Pasajeros por Terminal')
    plt.ylabel('Pasajeros')
    plt.show()

In [ ]:
# Boxplot por tipo de actividad
if 'activity_type_code' in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x='activity_type_code', y='passenger_count')
    plt.title('Distribución de Pasajeros por Tipo de Actividad')
    plt.xticks(rotation=30)
    plt.show()

In [ ]:
# Evolución temporal (si hay fecha)
if 'date' in df.columns:
    monthly = df.groupby(pd.Grouper(key='date', freq='M'))['passenger_count'].sum()
    plt.figure(figsize=(12,4))
    plt.plot(monthly.index, monthly.values)
    plt.title('Pasajeros Mensuales Totales')
    plt.ylabel('Pasajeros')
    plt.show()
    display(monthly.tail())

## 2. Segmentación de Aerolíneas (Clustering)
Se construyen variables agregadas por aerolínea y se aplica K-Means (selección de k por codo y validación con silhouette).

In [ ]:
# Ingeniería de características para clustering
cluster_cols_required = {'operating_airline','passenger_count'}
if cluster_cols_required.issubset(df.columns):
    feat = df.groupby('operating_airline').agg(
        total_pasajeros=('passenger_count','sum'),
        media_pasajeros=('passenger_count','mean'),
        registros=('passenger_count','count')
    )
    if 'geo_region' in df.columns:
        feat['destinos_unicos'] = df.groupby('operating_airline')['geo_region'].nunique()
    else:
        feat['destinos_unicos'] = 0
    if 'activity_type_code' in df.columns:
        feat['pct_enplaned'] = (df.groupby('operating_airline')['activity_type_code']
                                  .apply(lambda s: (s=='Enplaned').mean()))
        feat['pct_deplaned'] = (df.groupby('operating_airline')['activity_type_code']
                                  .apply(lambda s: (s=='Deplaned').mean()))
        feat['pct_transit'] = (df.groupby('operating_airline')['activity_type_code']
                                  .apply(lambda s: (s=='Transit').mean()))
    # Densidad (pasajeros / destinos)
    feat['pasajeros_por_destino'] = feat['total_pasajeros'] / feat['destinos_unicos'].replace(0, np.nan)
    feat.fillna(0, inplace=True)

    scaler = StandardScaler()
    Xc = scaler.fit_transform(feat)

    inertias = []
    silhouettes = []
    K_RANGE = range(2,9)
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = km.fit_predict(Xc)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(Xc, labels))

    fig, ax1 = plt.subplots(figsize=(8,4))
    ax1.plot(list(K_RANGE), inertias, marker='o', label='Inercia')
    ax1.set_xlabel('k'); ax1.set_ylabel('Inercia')
    ax2 = ax1.twinx()
    ax2.plot(list(K_RANGE), silhouettes, marker='s', color='orange', label='Silhouette')
    ax2.set_ylabel('Silhouette')
    plt.title('Método del Codo & Silhouette')
    plt.show()

    # Seleccionamos k óptimo (mayor silhouette razonable)
    k_opt = list(K_RANGE)[np.argmax(silhouettes)]
    print('k óptimo estimado:', k_opt)

    kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init='auto')
    feat['cluster'] = kmeans.fit_predict(Xc)

    # Proyección PCA para visualización
    pca = PCA(n_components=2, random_state=42)
    proj = pca.fit_transform(Xc)
    feat['pc1'] = proj[:,0]
    feat['pc2'] = proj[:,1]

    fig = px.scatter(feat.reset_index(), x='pc1', y='pc2', color='cluster', hover_name='operating_airline',
                     size='total_pasajeros', title='Aerolíneas Segmentadas (PCA)')
    fig.show()

    display(feat.groupby('cluster').agg(['mean','median']))
else:
    print('No se puede realizar clustering: faltan columnas necesarias.')

### 2.1 Interpretación de Clústeres (Personas)
Se crean descripciones cualitativas para cada segmento.

In [ ]:
if 'cluster' in feat.columns:
    personas = []
    for cid, g in feat.groupby('cluster'):
        desc = {
            'cluster': cid,
            'aerolineas': len(g),
            'pasajeros_tot_media': round(g['total_pasajeros'].mean()),
            'destinos_medios': round(g['destinos_unicos'].mean(),2),
            'perfil': ''
        }
        if g['total_pasajeros'].mean() > feat['total_pasajeros'].quantile(0.75):
            if g['destinos_unicos'].mean() > feat['destinos_unicos'].quantile(0.75):
                desc['perfil'] = 'Gigantes Internacionales'
            else:
                desc['perfil'] = 'Operadores de Alto Volumen Focalizados'
        elif g['total_pasajeros'].mean() < feat['total_pasajeros'].quantile(0.25):
            desc['perfil'] = 'Operadores de Nicho / Regionales'
        else:
            desc['perfil'] = 'Aerolíneas Medias Diversificadas'
        personas.append(desc)
    personas_df = pd.DataFrame(personas)
    display(personas_df)

## 3. Modelado de Regresión: Predicción de Pasajeros
Se predice `passenger_count` a partir de variables operativas.

In [ ]:
reg_required = {'passenger_count','operating_airline'}
if reg_required.issubset(df.columns):
    model_df = df.copy()

    if 'date' in model_df.columns:
        model_df['year'] = model_df['date'].dt.year
        model_df['month'] = model_df['date'].dt.month
        model_df['is_peak'] = model_df['month'].isin([6,7,8,12]).astype(int)
    else:
        model_df['year'] = 0
        model_df['month'] = 0
        model_df['is_peak'] = 0

    features = ['operating_airline','geo_region','terminal','activity_type_code','year','month','is_peak']
    features = [f for f in features if f in model_df.columns]

    X = model_df[features]
    y = model_df['passenger_count']

    cat_cols = [c for c in X.columns if X[c].dtype == 'object']
    num_cols = [c for c in X.columns if c not in cat_cols]

    preproc = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ])

    reg_pipe = Pipeline([
        ('prep', preproc),
        ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
    ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    reg_pipe.fit(X_train, y_train)
    y_pred = reg_pipe.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f'RMSE: {rmse:,.2f}\nMAE: {mae:,.2f}\nR2: {r2:,.4f}')

    # Importancias de características (aproximación)
    model = reg_pipe.named_steps['model']
    ohe = reg_pipe.named_steps['prep'].named_transformers_['cat'] if cat_cols else None
    feature_names = []
    if ohe is not None:
        feature_names.extend(list(ohe.get_feature_names_out(cat_cols)))
    feature_names.extend(num_cols)

    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
    display(importances.head(20))
    plt.figure(figsize=(8,6))
    importances.head(15).plot(kind='barh')
    plt.title('Importancia de Variables (Top 15)')
    plt.gca().invert_yaxis()
    plt.show()
else:
    print('No se puede entrenar regresión: faltan columnas necesarias.')

## 4. Clasificación: Tipo de Operativa Aérea
Modelo para predecir `activity_type_code`.

In [ ]:
class_required = {'activity_type_code','operating_airline'}
if class_required.issubset(df.columns):
    class_df = df.copy()
    if 'date' in class_df.columns:
        class_df['month'] = class_df['date'].dt.month
        class_df['year'] = class_df['date'].dt.year
    else:
        class_df['month'] = 0
        class_df['year'] = 0

    target = 'activity_type_code'
    features_c = [c for c in ['operating_airline','geo_region','terminal','year','month','passenger_count'] if c in class_df.columns]

    Xc = class_df[features_c]
    yc = class_df[target]

    cat_c = [c for c in Xc.columns if Xc[c].dtype == 'object']
    num_c = [c for c in Xc.columns if c not in cat_c]

    pre_c = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_c),
        ('num', 'passthrough', num_c)
    ])

    clf_pipe = Pipeline([
        ('prep', pre_c),
        ('model', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
    ])

    Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=42)
    clf_pipe.fit(Xc_train, yc_train)
    yc_pred = clf_pipe.predict(Xc_test)

    print(classification_report(yc_test, yc_pred))
    cm = confusion_matrix(yc_test, yc_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Matriz de Confusión')
    plt.ylabel('Real')
    plt.xlabel('Predicho')
    plt.show()
else:
    print('No se puede entrenar clasificación: faltan columnas necesarias.')

## 5. Series Temporales: Pronóstico Mensual
Se crea una serie mensual total y se entrena un modelo SARIMA (con opción auto-arima).

In [ ]:
if 'date' in df.columns and 'passenger_count' in df.columns:
    ts = df.groupby(pd.Grouper(key='date', freq='M'))['passenger_count'].sum().asfreq('M')
    ts = ts.fillna(0)

    # Descomposición
    decomposition = seasonal_decompose(ts, model='additive', period=12)
    decomposition.plot()
    plt.suptitle('Descomposición Serie Mensual')
    plt.show()

    # Test ADF
    adf_res = adfuller(ts)
    print(f'ADF Statistic: {adf_res[0]:.4f}, p-value: {adf_res[1]:.4f}')

    # Split train/test (últimos 12 meses para validación)
    if len(ts) > 24:
        train_ts = ts.iloc[:-12]
        test_ts = ts.iloc[-12:]
    else:
        train_ts = ts
        test_ts = ts.iloc[-1:]

    if PMDARIMA_AVAILABLE:
        auto_model = pm.auto_arima(train_ts, seasonal=True, m=12, trace=False, error_action='ignore', suppress_warnings=True)
        order = auto_model.order
        seasonal_order = auto_model.seasonal_order
        print('Orden seleccionado auto_arima:', order, seasonal_order)
    else:
        order = (1,1,1)
        seasonal_order = (1,1,1,12)

    sarima = SARIMAX(train_ts, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
    sarima_res = sarima.fit(disp=False)

    forecast_steps = 12 + 24  # 12 valid + 24 futuros
    forecast_res = sarima_res.get_forecast(steps=forecast_steps)
    fc = forecast_res.predicted_mean
    ci = forecast_res.conf_int()

    hist = ts
    plt.figure(figsize=(12,5))
    plt.plot(hist.index, hist.values, label='Histórico')
    plt.plot(fc.index, fc.values, label='Pronóstico', color='red')
    plt.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], color='pink', alpha=0.3, label='IC 95%')
    plt.axvline(test_ts.index[0], color='gray', linestyle='--', label='Inicio Validación')
    plt.title('Pronóstico Pasajeros Mensuales (SARIMA)')
    plt.legend()
    plt.show()

    # Métricas en ventana de validación
    overlap = fc.loc[test_ts.index]
    if len(overlap) == len(test_ts):
        val_rmse = mean_squared_error(test_ts, overlap, squared=False)
        val_mae = mean_absolute_error(test_ts, overlap)
        print(f'Validación - RMSE: {val_rmse:,.2f}  MAE: {val_mae:,.2f}')
else:
    print('No se puede construir serie temporal: faltan columnas date/passenger_count.')

## 6. Visualizaciones Resumen (Dashboard Simplificado)

In [ ]:
figs = []
if 'feat' in globals():
    fig_cluster = px.scatter(feat.reset_index(), x='pc1', y='pc2', color='cluster',
                             hover_data=['total_pasajeros','destinos_unicos'], title='Clusters Aerolíneas (PCA)')
    figs.append(fig_cluster)

if 'top_airlines' in globals():
    fig_air = px.bar(top_airlines.sort_values(ascending=True), orientation='h', title='Top 10 Aerolíneas')
    figs.append(fig_air)

if 'importances' in globals():
    fig_imp = px.bar(importances.head(15)[::-1], orientation='h', title='Importancia de Variables (Regresión)')
    figs.append(fig_imp)

if 'fc' in globals() and 'ts' in globals():
    hist_df = pd.DataFrame({'fecha': ts.index, 'pasajeros': ts.values})
    fc_df = pd.DataFrame({'fecha': fc.index, 'pronostico': fc.values})
    fig_ts = go.Figure()
    fig_ts.add_trace(go.Scatter(x=hist_df['fecha'], y=hist_df['pasajeros'], name='Histórico'))
    fig_ts.add_trace(go.Scatter(x=fc_df['fecha'], y=fc_df['pronostico'], name='Pronóstico'))
    fig_ts.update_layout(title='Serie Temporal y Pronóstico')
    figs.append(fig_ts)

for f in figs:
    f.show()

## 7. Recomendaciones de Negocio (Borrador)
Basado en resultados (ajustar tras revisión de datos reales):

1. Optimización de Gate/Terminal: Segmentar asignaciones priorizando clúster de "Gigantes Internacionales" en horarios pico y redistribuir slots a operadores de nicho fuera de picos para suavizar la curva de demanda.
2. Planificación de Recursos: El modelo de regresión indica que variables como [reemplazar con top features reales] explican gran parte de la variabilidad; enfocar staffing dinámico según esas señales.
3. Estrategia de Crecimiento: Pronóstico SARIMA sugiere tendencia + estacionalidad marcada en meses de verano; preparar capacidad adicional (personal, mantenimiento flexible) 2 meses antes de picos.
4. Diversificación de Rutas: Clústeres con baja diversificación de destinos podrían beneficiarse de incentivos para abrir rutas internacionales medianas, equilibrando riesgo y utilización de infraestructura.

(Ajustar texto con números concretos tras validación final.)

## 8. Próximos Pasos
- Validar supuestos sobre columnas (si nombres difieren, adaptar mapa de renombrado).
- Añadir modelo alternativo (Gradient Boosting / XGBoost) y comparar.
- Incluir explicabilidad SHAP para regresión (si dataset final no es demasiado grande).
- Agregar visualización geoespacial (Folium) con rutas principales.

---
Fin del Notebook.